In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

print("Carregando datasets...")


data = pd.read_csv('datasets/Breast_GSE70947.csv')
data.sample(5)


Carregando datasets...


,samples,type,NM_144987,NM_013290,ENST00000322831,NM_001625,lincRNA:chr7:226042-232442_R,NM_032391,ENST00000238571,XR_108906,...,lincRNA:chr4:77860976-77869926_F,NM_152343,NM_001005327,NM_001039355,lincRNA:chr21:44456656-44468556_R,lincRNA:chr9:4869500-4896050_F,NM_016053,NM_001080425,ENST00000555638,ENST00000508993
144,GSM1823848_252800416912_S01_GE1_107_Sep09_2_1,normal,8.776873,7.930760,7.280372,11.235364,9.481026,5.054622,6.127273,4.884584,...,7.573449,6.356468,4.370383,10.361963,5.137988,6.211367,10.872875,8.587266,4.501017,4.489114
54,GSM1823757_252800416868_S01_GE1_107_Sep09_1_1,normal,9.104604,7.710008,6.237543,11.819201,8.871537,5.126962,5.883678,5.410166,...,6.988327,6.260468,4.326804,11.660564,4.718107,6.137760,10.355549,9.088977,4.539635,4.911437
221,GSM1823930_252800416875_S01_GE1_107_Sep09_1_2,breast_adenocarcinoma,9.836164,9.111102,7.033582,11.706936,8.065982,5.030828,5.915396,4.712845,...,6.239594,5.867454,4.663480,12.023106,5.001146,4.871339,10.181151,9.719479,5.030828,5.229057
37,GSM1823739_252800416858_S01_GE1_107_Sep09_1_3,normal,8.876949,6.921600,6.385790,11.129354,8.581399,4.654774,6.634849,4.512760,...,6.345094,6.029624,4.810068,12.597862,5.004462,6.330783,10.524779,9.105153,5.020834,4.795166
167,GSM1823871_111101_252800411732_S01_GE1_107_Sep...,breast_adenocarcinoma,8.146231,7.636391,5.965391,11.131801,9.883508,4.491383,5.892747,4.702040,...,6.947437,6.117530,4.445481,11.002289,4.959261,6.141411,10.621860,7.896673,5.006064,4.981512


In [19]:
data['type'].value_counts()

type
normal                   146
breast_adenocarcinoma    143
Name: count, dtype: int64

In [29]:
# Separa X (features) e y (target)
X = data.drop(['type', 'samples'], axis=1)
y = data['type'].replace({'normal': 0, 'breast_adenocarcinoma': 1})  # Codifica as classes como 0 e 1

print(f"Dataset carregado: {X.shape[0]} amostras e {X.shape[1]} genes (features).")
print(f"Distribuição de classes: \n{y.value_counts()}")

# ==========================================
# 2. Divisão Treino/Teste (Holdout)
# ==========================================
# Separamos 20% para o teste final (que o modelo nunca verá durante o CV)
# O parâmetro stratify garante que a proporção tumor/normal seja mantida
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30, 
    stratify=y, 
    random_state=42
)

# ==========================================
# 3. Normalização
# ==========================================
# Importante: O fit (cálculo da média/desvio) é feito APENAS no treino
# para evitar vazamento de dados (data leakage) para o teste.
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# ==========================================
# 4. K-Fold Estratificado e Validação
# ==========================================
print("\nIniciando Treinamento com Cross-Validation (K-Fold)...")

# Configura o Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,  # Número de árvores
    random_state=42,
    n_jobs=-1          # Usa todos os processadores
)

# Configura o K-Fold Estratificado (Ex: 5 pastas)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Avalia usando F1-Score no conjunto de TREINO (Validação Cruzada)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=kfold, scoring='f1')

print(f"Scores F1 em cada fold: {cv_scores}")
print(f"Média do F1-Score na Validação Cruzada: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# ==========================================
# 5. Avaliação Final (Conjunto de Teste)
# ==========================================
print("\nAvaliação Final no Dataset de Teste (Holdout)...")

# Treina o modelo com TODO o conjunto de treino
rf_model.fit(X_train, y_train)

# Predição no teste
y_pred = rf_model.predict(X_test)

# Métrica Final
final_f1 = f1_score(y_test, y_pred)

print(f"F1-Score Final (Teste): {final_f1:.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred, target_names=['normal', 'breast_adenocarcinoma']))

# Opcional: Mostra as features mais importantes (Genes)
feature_importances = pd.DataFrame(rf_model.feature_importances_,
                                    index = X.columns,
                                    columns=['importance']).sort_values('importance', ascending=False)
print(feature_importances.head(10))

C:\Users\grkremer\AppData\Local\Temp\ipykernel_7624\3198102146.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = data['type'].replace({'normal': 0, 'breast_adenocarcinoma': 1})  # Codifica as classes como 0 e 1


Dataset carregado: 289 amostras e 35981 genes (features).
Distribuição de classes: 
type
0    146
1    143
Name: count, dtype: int64

Iniciando Treinamento com Cross-Validation (K-Fold)...
Scores F1 em cada fold: [0.9        0.95238095 0.75675676 0.6875     0.85      ]
Média do F1-Score na Validação Cruzada: 0.8293 (+/- 0.1917)

Avaliação Final no Dataset de Teste (Holdout)...
F1-Score Final (Teste): 0.8941

Relatório de Classificação Detalhado:
                       precision    recall  f1-score   support

               normal       0.89      0.91      0.90        44
breast_adenocarcinoma       0.90      0.88      0.89        43

             accuracy                           0.90        87
            macro avg       0.90      0.90      0.90        87
         weighted avg       0.90      0.90      0.90        87

              importance
NM_002263       0.010796
NM_207311       0.010194
NM_144497       0.010053
NR_027134       0.009263
NM_001024401    0.009151
NM_152515       0.0